# Downstream analysis: AnnData + Pandas + NumPy + Scikit-learn
รัน **Restart Kernel and Run All** ได้ด้วยข้อมูลสังเคราะห์ที่กำหนด seed ไม่มีการดาวน์โหลดข้อมูล
เปลี่ยน `COUNTS_PATH` เพื่อใช้ matrix จาก Bash หรือ Nextflow (แถว = genes, คอลัมน์ = cells)
ตัวอย่างนี้สาธิตการทำงานของซอฟต์แวร์ ไม่ใช่ข้อมูลสำหรับสรุปชีววิทยา

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Works from repository root, workbench/, or workbench/notebooks/.
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'sc_workbench').is_dir() or (p / 'workbench/sc_workbench').is_dir())
package_root = root if (root / 'sc_workbench').is_dir() else root / 'workbench'
sys.path.insert(0, str(package_root))
from sc_workbench import io, qc, preprocess, reduction, clustering, markers, SingleCellWorkbench
sc.settings.verbosity = 0
GTF_PATH = None  # Set to the exact upstream genes.gtf when using Ensembl IDs
COUNTS_PATH = None  # e.g. Path('/absolute/path/results/counts/gene_cell_count_matrix.tsv')
OUTPUT = package_root / 'results' / 'tutorial'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Input และ metadata
Pandas/NumPy ใช้ cells × genes; upstream TSV ใช้ genes × cells และ loader จะ transpose ให้
ชื่อเซลล์ต้องไม่ซ้ำ ใช้ index ในการจับคู่ metadata เสมอ
สำหรับ Ensembl IDs ให้ใส่ gene symbols จาก GTF รุ่นเดียวกันใน `adata.var['gene_name']` ก่อนคำนวณ mitochondrial QC

In [2]:
if COUNTS_PATH is None:
    rng = np.random.default_rng(42)
    truth = np.repeat(['A', 'B', 'C'], 40)
    rates = np.full((120, 300), 1.5)
    for i in range(3):
        rates[i*40:(i+1)*40, 10+i*30:40+i*30] = 9
    df = pd.DataFrame(rng.poisson(rates),
                      index=[f'cell_{i:03}' for i in range(120)],
                      columns=['MT-DEMO'] + [f'gene_{i}' for i in range(299)])
    adata = io.from_dataframe(df)
    adata.obs['synthetic_group'] = truth
else:
    adata = io.load_upstream_matrix(COUNTS_PATH)
if GTF_PATH is not None:
    io.annotate_from_gtf(adata, GTF_PATH)
    print('Matched annotation:', adata.var['annotation_matched'].mean())
# Example real metadata: adata.obs = metadata.loc[adata.obs_names].copy()
adata

✔ Ingested DataFrame: 120 cells × 300 genes

AnnData object with n_obs × n_vars = 120 × 300
    obs: 'synthetic_group'
    var: 'gene_name'
    layers: None (.X), 'counts'

## 2. QC และ filtering
ตรวจ distribution แล้วเลือก threshold ตาม protocol ตัวเลขนี้มีไว้สำหรับ demo เท่านั้น
ไม่เรียก Scrublet อัตโนมัติสำหรับ plate-based Smart-seq2; พิจารณาวิธีตรวจ doublet ตามการเตรียมตัวอย่าง
QC ใช้ `layers['counts']` แม้ `.X` ผ่าน normalization แล้ว

In [3]:
qc.calculate_qc_metrics(adata)
display(adata.obs[['total_counts', 'n_genes_by_counts', 'pct_counts_mito']].describe())
adata.obs[['total_counts', 'n_genes_by_counts', 'pct_counts_mito']].hist(bins=20, figsize=(10, 3))
adata = qc.filter_cells(adata, min_genes=20, min_counts=50, max_pct_mito=30)
qc.filter_genes(adata, min_cells=3)
assert adata.n_obs > 3 and adata.n_vars > 3, 'Inspect QC thresholds: too few cells/genes remain' 

QC Calculated: Median counts: 677 | Median genes: 239 | Median Mito %: 0.15%

,total_counts,n_genes_by_counts,pct_counts_mito
count,120.000000,120.000000,120.000000
mean,676.616638,239.433333,0.203519
std,25.825264,7.065995,0.163112
min,614.000000,218.000000,0.000000
25%,661.500000,234.000000,0.139912
50%,677.000000,239.000000,0.149365
75%,692.000000,245.000000,0.297509
max,748.000000,255.000000,0.690608


✔ Filtered Cells: Retained 120 cells (dropped 0 / 120, 100.0% retained)

✔ Filtered Genes: Retained 300 / 300 genes (min_cells >= 3)

## 3. Normalization → HVG → PCA → neighbors → UMAP → Leiden
ฟังก์ชันส่วนใหญ่แก้ AnnData เดิมและคืน object เดิม; `filter_cells` คืนสำเนาที่กรองแล้ว
`counts` = original counts, `normalized` และ `.raw` = log1p normalized expression
เก็บ genes ทั้งหมดไว้สำหรับ markers; PCA เลือกเฉพาะ HVG ไม่จำเป็นต้อง scale ทุก gene จน dense
[Scanpy HVG documentation](https://scanpy.readthedocs.io/en/stable/api/scanpy.pp.highly_variable_genes.html): `seurat` ใช้ log expression ส่วน `seurat_v3` ใช้ counts และต้องติดตั้ง scikit-misc เพิ่ม

In [4]:
preprocess.normalize_and_log(adata)
preprocess.select_hvg(adata, n_top_genes=min(150, adata.n_vars), flavor='seurat')
reduction.run_pca(adata, n_comps=20)
reduction.compute_neighbors(adata, n_neighbors=10, n_pcs=20, use_rep='X_pca')
reduction.run_umap(adata, random_state=42)
clustering.cluster_leiden(adata, resolution=0.5)
sc.pl.umap(adata, color=['leiden', 'total_counts'], show=True)

Normalizing counts to 10,000 and computing log1p...

✔ Normalization & log1p complete.

Selecting top 150 Highly Variable Genes (flavor=seurat)...

✔ HVG Selection: Identified 150 highly variable genes.

Computing PCA (20 components)...

...

Computing 2D UMAP non-linear manifold projection...

✔ UMAP coordinates computed (stored in .obsm['X_umap']).

Clustering cells with Leiden (resolution=0.5)...

✔ Leiden clustering complete: Discovered 3 clusters.

/home/surj/Workspace/single_cell_pipeline/workbench/.pixi/envs/default/lib/python3.12/site-packages/scanpy/plotting/_utils.py:393: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. ใช้ Pandas / NumPy / Scikit-learn โดยตรง
ไม่จำเป็นต้องห่อ estimator ทุกตัวใน workbench ใช้ embedding ที่เก็บใน `.obsm` ได้ทันที
ใช้ sparse matrix ผ่าน `.layers` สำหรับ matrix ใหญ่; `.to_numpy()` / `.to_df()` จะแปลงเป็น dense
ARI ด้านล่างเทียบกับ label สังเคราะห์เท่านั้น ไม่ใช่ validation ของ cell type จริง
หากฝึก classifier จริง ให้แยก train/test ตาม donor และ fit preprocessing เฉพาะ train เพื่อป้องกัน leakage

In [5]:
pca = io.get_embedding_df(adata, 'X_pca')
adata.obs['kmeans'] = pd.Categorical(KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(pca).astype(str))
display(pd.crosstab(adata.obs['leiden'], adata.obs['kmeans']))
if 'synthetic_group' in adata.obs:
    print('Synthetic ARI:', adjusted_rand_score(adata.obs['synthetic_group'], adata.obs['kmeans']))
# Inspect a small dense slice only
expression = io.to_dataframe(adata[:, :5], layer='normalized')
display(expression.head())
print('Counts per cell:', np.asarray(adata.layers['counts'].sum(axis=1)).ravel()[:5])

kmeans,0,1,2
leiden,,,
0,0,0,40
1,40,0,0
2,0,40,0


Synthetic ARI: 1.0


,MT-DEMO,gene_0,gene_1,gene_2,gene_3
cell_000,3.834037,0.000000,3.834037,2.777760,3.834037
cell_001,0.000000,3.815043,2.759561,2.759561,0.000000
cell_002,0.000000,2.758176,2.758176,3.419106,2.758176
cell_003,2.760947,0.000000,3.421968,0.000000,2.760947
cell_004,3.465978,3.465978,2.803595,0.000000,0.000000


Counts per cell: [663. 676. 677. 675. 645.]


## 5. Marker exploration และ checkpoint
ใช้ log normalized expression สำหรับ marker tests ไม่ใช้ scaled matrix
cluster markers เป็น exploratory; การเปรียบเทียบ treatment ต้องคำนึงถึง biological replicates
อย่าตั้งชื่อ cell type จากหมายเลข cluster เพียงอย่างเดียว ต้องตรวจ marker และบริบท tissue

In [6]:
if adata.obs['leiden'].nunique() > 1:
    markers.find_markers(adata, groupby='leiden', n_genes=10)
    marker_table = markers.get_markers_df(adata)
    display(marker_table.groupby('cluster', observed=True).head(3))
    marker_table.to_csv(OUTPUT / 'markers.csv', index=False)
io.save_h5ad(adata, OUTPUT / 'processed.h5ad')
adata.obs.to_csv(OUTPUT / 'cell_metadata.csv')
pca.to_csv(OUTPUT / 'pca.csv')
loaded = sc.read_h5ad(OUTPUT / 'processed.h5ad')
assert loaded.shape == adata.shape
assert 'counts' in loaded.layers

Finding cluster marker genes using wilcoxon test on 'leiden'...

✔ Marker gene discovery complete.

,cluster,gene,score,logfoldchange,pvals,pvals_adj,pct_nz_group,pct_nz_reference
0,0,gene_10,8.907235,3.422396,5.232087e-19,1.930745e-17,1.0,0.8000
1,0,gene_9,8.907235,3.307450,5.232087e-19,1.930745e-17,1.0,0.8125
2,0,gene_28,8.896101,4.198659,5.784274e-19,1.930745e-17,1.0,0.6500
10,1,gene_59,8.901669,3.593447,5.501341e-19,2.041843e-17,1.0,0.7625
11,1,gene_66,8.901669,3.026091,5.501341e-19,2.041843e-17,1.0,0.8625
12,1,gene_39,8.890534,3.512815,6.081572e-19,2.041843e-17,1.0,0.7750
20,2,gene_73,8.907235,3.659012,5.232087e-19,2.104946e-17,1.0,0.7500
21,2,gene_89,8.907235,3.473410,5.232087e-19,2.104946e-17,1.0,0.8000
22,2,gene_75,8.907235,3.933658,5.232087e-19,2.104946e-17,1.0,0.7000


✔ Saved AnnData to: /home/surj/Workspace/single_cell_pipeline/workbench/results/tutorial/processed.h5ad

## 6. Optional fluent API
เป็น wrapper บาง ๆ บน AnnData เท่านั้น ใช้ `wb.adata` ร่วมกับ Scanpy ได้
constructor รับ object เดิม หากต้องการแยกการแก้ไขให้ `.copy()` ก่อน

In [7]:
wb = SingleCellWorkbench(adata.copy())
wb.add_obs(pd.Series('reviewed', index=wb.obs.index[::-1]), col_name='review_status')
display(wb.get_embedding('X_umap').head())
wb

,UMAP_1,UMAP_2
cell_000,10.156486,7.855716
cell_001,9.847693,9.227558
cell_002,9.897025,8.393405
cell_003,9.897568,8.989580
cell_004,9.100796,7.980886


<SingleCellWorkbench: 120 cells × 300 genes>